# Ethical principle properties

`PGD-Robustness.ipynb` showed that PGD is the wrong tool for this network: the correct effort
changes inside the epsilon-ball, so "the output must not change" is not the right requirement.
This notebook takes the next step: stating the SaveMe ethical principles as **properties** of the
trained network, verifying which of them hold with Vehicle, and using the counterexamples to see
where and why the others fail.

The controller that generated the labels (`simulation.py`) scores a room by its expected utility,
so P1 holds for it by construction. The other principles are deliberately **not** part of the
controller, so verifying them tests what the network learned rather than what it was told.

Stages:

1. **Model**: train the base network and export it to ONNX for Vehicle
2. **Ethical principles**: state a principle as a property of `effort`, with P1 as a worked example
3. **Verification**: check the property against the network with Vehicle and Marabou
4. **Coursework**: specify and verify all nine principles, and test whether they conflict

In [1]:
#####################################
### Import the relevant libraries ###
#####################################

import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import json
import os

import onnx
from onnx import helper, numpy_helper, TensorProto
from onnx.reference import ReferenceEvaluator

To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


***
# **DATA**

In [2]:
#########################################
### Load the moral regression dataset ###
#########################################

# Each row is one timestep of a rescue episode: the agent's accumulated risk R and the
# room it has reached (5 inputs), followed by the regression target `effort` in [0, 1]
features = ['R', 'p_alive', 'n_estimated', 'urgency', 'is_closed']

def load_split(split):
    data = np.loadtxt(f'data/{split}.csv', delimiter=',', skiprows=1, dtype='float32')
    return data[:, :5], data[:, 5]

X_train, y_train = load_split('train')
X_val, y_val = load_split('val')
X_test, y_test = load_split('test')

# The controller constants (TAU, K_EFFORT, R_MAX, ...), needed later as Vehicle parameters
with open('data/meta.json') as f:
    meta = json.load(f)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print({k: meta[k] for k in ['TAU', 'K_EFFORT', 'R_MAX', 'RISK_RECOVERY', 'RISK_PER_EFFORT']})

(14000, 5)
(3000, 5)
(3000, 5)
{'TAU': 5.0, 'K_EFFORT': 0.05, 'R_MAX': 1.0, 'RISK_RECOVERY': 0.2, 'RISK_PER_EFFORT': 1.0}


***
# **MODEL**

In [3]:
##########################
### Set some variables ###
##########################

input_size = 5
batch_size = 64
epochs = 50
seed = 0

In [4]:
########################
### Define the model ###
########################

# The same network as in PGD-Robustness.ipynb: Dense and ReLU layers only, which is what
# Vehicle's verifier backends support
def get_model(input_size, seed=42):
    initializer = tf.keras.initializers.GlorotUniform(seed=seed)
    model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_size,), name='input_features'),
            tf.keras.layers.Dense(64, activation='relu', kernel_initializer=initializer, name='dense_1'),
            tf.keras.layers.Dense(64, activation='relu', kernel_initializer=initializer, name='dense_2'),
            tf.keras.layers.Dense(1, activation='linear', kernel_initializer=initializer, name='output_layer')
        ])
    print(model.summary())
    return model

In [5]:
##########################
### Train a base model ###
##########################

# Fix the initialisation and the shuffling so the verified network is reproducible
tf.keras.utils.set_random_seed(seed)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train[:, None]))
train_dataset = train_dataset.shuffle(buffer_size=1024, seed=seed).batch(batch_size)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val[:, None])).batch(batch_size)

model_base = get_model(input_size, seed=seed)
model_base.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.MeanSquaredError())
model_base.fit(train_dataset, epochs=epochs, validation_data=val_dataset, verbose=2)

# Test accuracy
y_pred = model_base.predict(X_test, verbose=0)[:, 0]
mse = np.mean((y_pred - y_test) ** 2)
r2 = 1 - np.sum((y_pred - y_test) ** 2) / np.sum((y_test - y_test.mean()) ** 2)
print(f'Test MSE: {mse:.6f}')
print(f'Test R2: {r2:.4f}')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,609 (18.00 KB)

 Trainable params: 4,609 (18.00 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/50


/home/melon501/source/JC-DAIR/moral-regression/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


219/219 - 1s - 3ms/step - loss: 0.0050 - val_loss: 0.0017
Epoch 2/50
219/219 - 0s - 747us/step - loss: 0.0015 - val_loss: 0.0018
Epoch 3/50
219/219 - 0s - 752us/step - loss: 0.0011 - val_loss: 7.5315e-04
Epoch 4/50
219/219 - 0s - 734us/step - loss: 7.3089e-04 - val_loss: 7.2544e-04
Epoch 5/50
219/219 - 0s - 759us/step - loss: 5.6572e-04 - val_loss: 4.3306e-04
Epoch 6/50
219/219 - 0s - 734us/step - loss: 5.6720e-04 - val_loss: 4.0046e-04
Epoch 7/50
219/219 - 0s - 788us/step - loss: 4.3796e-04 - val_loss: 2.9263e-04
Epoch 8/50
219/219 - 0s - 763us/step - loss: 3.3343e-04 - val_loss: 4.1161e-04
Epoch 9/50
219/219 - 0s - 634us/step - loss: 2.6445e-04 - val_loss: 2.6178e-04
Epoch 10/50
219/219 - 0s - 638us/step - loss: 2.3259e-04 - val_loss: 1.6011e-04
Epoch 11/50
219/219 - 0s - 678us/step - loss: 1.5025e-04 - val_loss: 1.5131e-04
Epoch 12/50
219/219 - 0s - 654us/step - loss: 1.6727e-04 - val_loss: 1.7917e-04
Epoch 13/50
219/219 - 0s - 666us/step - loss: 1.3778e-04 - val_loss: 1.0700e-04
Ep

In [6]:
######################
### Save the model ###
######################

os.makedirs('models', exist_ok=True)
model_base.save('models/base.keras')

***
# **EXPORT TO ONNX**

Vehicle reads networks in ONNX format. The network is written directly as a chain of `Gemm`
(matrix multiply plus bias) and `Relu` nodes, one per Dense layer. The inputs were never
normalised, so the ONNX network takes raw `[R, p_alive, n_estimated, urgency, is_closed]` and the
properties can be written in the same units as the data, with no normalisation step in the spec.

In [7]:
#######################################
### Define the ONNX export function ###
#######################################

def export_onnx(model, path):
    nodes, initializers, current = [], [], 'input'
    dense_layers = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Dense)]

    for i, layer in enumerate(dense_layers):
        W, b = layer.get_weights()
        initializers += [numpy_helper.from_array(W.astype(np.float32), f'W{i}'),
                         numpy_helper.from_array(b.astype(np.float32), f'b{i}')]

        # Dense layer: z = x @ W + b
        is_last = i == len(dense_layers) - 1
        z = 'output' if is_last else f'z{i}'
        nodes.append(helper.make_node('Gemm', [current, f'W{i}', f'b{i}'], [z]))
        current = z

        # ReLU on the hidden layers (the output layer is linear)
        if layer.get_config()['activation'] == 'relu':
            nodes.append(helper.make_node('Relu', [z], [f'a{i}']))
            current = f'a{i}'

    graph = helper.make_graph(
        nodes, 'effort_network',
        [helper.make_tensor_value_info('input', TensorProto.FLOAT, [1, input_size])],
        [helper.make_tensor_value_info('output', TensorProto.FLOAT, [1, 1])],
        initializers)
    onnx_model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 13)])
    onnx.checker.check_model(onnx_model)
    onnx.save(onnx_model, path)

In [8]:
###################################################
### Export the model and check it matches Keras ###
###################################################

export_onnx(model_base, 'models/base.onnx')

# Run the ONNX network on every test row and compare with the Keras predictions
session = ReferenceEvaluator('models/base.onnx')
y_onnx = np.array([session.run(None, {'input': x[None, :]})[0][0, 0] for x in X_test])
print(f'Largest difference between ONNX and Keras on the test set: {np.max(np.abs(y_onnx - y_pred)):.2e}')

Largest difference between ONNX and Keras on the test set: 2.98e-07


***
# **ETHICAL PRINCIPLES**

SaveMe scores a room with ethical principles. Three are known so far, written as *score terms*:

| principle | idea | SaveMe term |
|---|---|---|
| P1 | maximise expected utility | `3 · n_estimated · p_alive · urgency` |
| P7 | defer to certainty | `8 · (1 − p_alive)` if closed, else 0 |
| P9 | avoid likely failure | `30 · (0.3 − p_alive · rs)` if `p_alive · rs < 0.3`, else 0 |

with `rs` = 0.50 / 0.70 / 0.95 for urgency 5 / 3 / 1.

For verification, each principle has to become a statement that is **true or false** about the
network's `effort` over a region of inputs, rather than a term in a score. This is how the ACAS Xu
specification works: each of its properties picks a region of situations (e.g. "the intruder is
distant and significantly slower than the ownship") and bounds what the network may output there.

### Worked example: P1 as a property

**Expected utility** is `EU = n_estimated · p_alive · urgency`. That is a product of inputs, which a
linear verifier such as Marabou cannot handle directly. So, like the ACAS Xu regions, the room is
described by **lower bounds on each factor**, which together guarantee a lower bound on EU:

```
highExpectedUtility x = n_estimated ≥ 4  and  p_alive ≥ 0.8  and  urgency == 5      -- so EU ≥ 16
lowRisk x             = R ≤ 0.2                                                      -- risk budget left

minimumEffort = kEffort · (16 − tau) · (1 − 0.2 / rMax)                              -- = 0.44
```

**P1 (maximise expected utility)**: if a room's expected utility is certainly high and the agent is
not at risk, the agent commits at least the effort that expected utility warrants.

```
forall x . validInput x and highExpectedUtility x and lowRisk x =>
  effort x ! effortLevel >= minimumEffort - tolerance
```

`validInput` rules out rooms that cannot exist: an open room must have `p_alive = 1`, and `urgency`
must be 1, 3 or 5. The controller constants (`tau`, `kEffort`, `rMax`) are passed in from
`meta.json`. `tolerance` allows for the network only *approximating* the controller.

The controller's effort increases with `n_estimated`, `p_alive` and `urgency` and decreases with
`R`, so its lowest effort in the region is at the corner (n = 4, p_alive = 0.8, R = 0.2), where it is
exactly 0.44. **P1 therefore holds for the controller by construction.** The question is whether it
holds for the network.

The full specification is in `principles.vcl`:

In [9]:
######################################
### Show the Vehicle specification ###
######################################

print(open('principles.vcl').read())

--------------------------------------------------------------------------------
-- Ethical principle properties of the SaveMe effort network

-- The network is trained on data from `simulation.py`, whose controller is
--
--   effort = kEffort * max(0, score - tau) * max(0, 1 - R / rMax)
--   score  = n_estimated * p_alive * urgency
--
-- The SaveMe principles are NOT part of that controller. Each property below
-- states one principle as a statement about the network's effort, so that
-- verifying it tells us which principles the trained network embodies.

--------------------------------------------------------------------------------
-- Inputs

-- The network takes inputs of the form of a vector of 5 numbers.

type InputVector = Tensor Real [5]

-- Meaningful names for the indices, in the column order of the data.

risk       = 0   -- R, the agent's accumulated risk before this room
pAlive     = 1   -- probability the children in the room are alive
nEstimated = 2   -- estimated numb

***
# **VERIFICATION**

Vehicle compiles the property into queries for the Marabou verifier and runs them against
`models/base.onnx`. Marabou either proves that no input in the region breaks the property, or
returns a **counterexample**: an input where it does.

The cells below run `verify.sh` as a bash script, exactly as you would from a terminal:
`./verify.sh [network.onnx] [tolerance]`. It calls `vehicle verify` with the controller constants
from `meta.json` (`TAU = 5.0`, `K_EFFORT = 0.05`, `R_MAX = 1.0`), using `vehicle` and Marabou from
this folder's `.venv`. P1 is verified twice: strictly (`tolerance = 0`), and allowing the
network to fall up to 0.05 below the effort the controller guarantees.

In [10]:
%%bash
cat verify.sh

#!/bin/bash
# Verify the ethical principle properties in principles.vcl with Vehicle + Marabou.
#
#   ./verify.sh [network.onnx] [tolerance]
#
# Defaults: models/base.onnx and tolerance 0.05. The controller constants come from data/meta.json.

set -e
cd "$(dirname "$0")"

# Use vehicle and Marabou from this folder's .venv, whether or not it is activated
if [ -d .venv/bin ]; then
  PATH="$(cd .venv/bin && pwd):$PATH"
fi

NETWORK=${1:-models/base.onnx}
TOLERANCE=${2:-0.05}

vehicle verify \
  -v Marabou \
  -s principles.vcl \
  -n effort:$NETWORK \
  -p tau:5.0 \
  -p kEffort:0.05 \
  -p rMax:1.0 \
  -p tolerance:$TOLERANCE


In [11]:
%%bash
# Verify P1 strictly: tolerance = 0
./verify.sh models/base.onnx 0



In order to provide support, Vehicle has automatically converted the strict inequalities to non-strict inequalites. This is not sound, but errors will be at most the floating point epsilon used by the solver, which is usually very small (e.g. 1e-9). However, this may lead to unexpected behaviour (e.g. loss of the law of excluded middle).

See https://github.com/vehicle-lang/vehicle/issues/74 for further details.



  maximiseExpectedUtility [========================================] 6/6 queries
Verifying properties:
    result: ✗ - Marabou found a counterexample
      x: [ 0.2, 0.8, 4.0, 5.0, 1.0 ]


**With no tolerance, P1 fails.** Marabou returns the counterexample `x: [ 0.2, 0.8, 4.0, 5.0, 1.0 ]`,
which is `R = 0.2, p_alive = 0.8, n_estimated = 4, urgency = 5` in a closed room.

This is the region's **least favourable corner**: the lowest expected utility (`n_estimated` and
`p_alive` at their lower bounds) and the highest risk (`R` at its upper bound). It is exactly where
the controller's effort is lowest. The controller gives 0.44 there, just meeting the bound, and the
network gives about 0.41, slightly below it.

So even a property that holds **by construction** for the controller can fail for the network
trained on it. The network only approximates the controller, and the verifier searches the whole
region, so it finds the one corner where a small approximation error pushes the network below the
bound. Testing on sampled data would very likely miss it: the corner is a single point.

In [12]:
%%bash
# Verify P1 allowing the network 0.05 below the controller: tolerance = 0.05
./verify.sh models/base.onnx 0.05



In order to provide support, Vehicle has automatically converted the strict inequalities to non-strict inequalites. This is not sound, but errors will be at most the floating point epsilon used by the solver, which is usually very small (e.g. 1e-9). However, this may lead to unexpected behaviour (e.g. loss of the law of excluded middle).

See https://github.com/vehicle-lang/vehicle/issues/74 for further details.



  maximiseExpectedUtility [========================================] 6/6 queries
Verifying properties:
    result: 🗸 - Marabou proved no counterexample exists


**With a tolerance of 0.05, P1 holds.** Marabou proves that for *every* input in the region, not
just the test rows, the network's effort is at most 0.05 below what the controller guarantees.
The tolerance is part of the claim: the network satisfies P1 up to an approximation error of 0.05.
Try smaller tolerances in `verify.sh` to find how tight the guarantee can be made.

The warning about strict inequalities is expected. To search for a counterexample, Vehicle negates
the property, which turns its `>=` into `<`, and Marabou only supports `<=`. The error this
introduces is at most the solver's floating-point epsilon.

***
# **COURSEWORK**

P1 is one of **nine** ethical principles in the search and rescue setting. Read the appendix of
[Interpretable Moral Decision-making under Epistemic Uncertainty (Ciupa et al., NeSy 2026)](https://www.pure.ed.ac.uk/ws/portalfiles/portal/660548025/CiupaEtalNESY2026InterpretableMoralDecision-making.pdf)
for the definitions of all nine principles, P1 to P9.

1. **Specify the principles.** Write each principle as one or more properties in `principles.vcl`,
   in the same style as `maximiseExpectedUtility` and the ten ACAS Xu properties: name the region
   of inputs with predicates, derive the bound on `effort`, and explain both in comments. Decide
   whether each property needs a `tolerance`.

2. **Verify them.** Check each property against `models/base.onnx` with `verify.sh`, in a `%%bash`
   cell as above or from a terminal. To check one property at a time, add `-y <propertyName>` to
   the `vehicle verify` command in `verify.sh`. For every counterexample, explain where it lies in
   the region and why, as was done for P1 above.

3. **Do the conflicts survive formal verification?** Appendix A1 of the paper reports a key conflict
   between the ethical principles. Does that conflict also hold under formal verification?
   Train a network with **property-driven training** on P1 (covered in future lectures) and verify
   P7 against it. We would expect P7 to fail. Then do the reverse: train on P7 and
   verify P1. Does this happen in practice? Use the counterexamples to explain what you find.